# 위험 소리 분류 파인튜닝 Colab

이 노트북은 회의록/중간보고서에 적힌 후보 모델과 데이터셋을 기준으로, 청각장애인 지원 키링/앱 서비스에서 중요한 **위험·주의 소리**를 우선 분류하도록 구성했습니다.

- 기본 타깃 클래스: `car_horn`, `siren`, `glass_breaking`, `explosion_or_gunshot`, `construction_or_machine`, `fire_alarm`, `baby_cry`, `doorbell_or_bell`, `other`
- 기본 자동 다운로드: ESC-50
- 선택 다운로드/로딩: UrbanSound8K, FSD50K, AudioSet, AI Hub 소음 환경 음성인식 데이터
- 위험 소리 성능 보강: 클래스 가중치, `other` 샘플 제한, augmentation, validation F1 확인

> Colab에서 먼저 `런타임 > 런타임 유형 변경 > GPU`를 선택하세요. 빠른 실험은 `EPOCHS=3`, `MAX_SAMPLES_PER_CLASS=300` 정도로 시작하고, 최종 비교 때 늘리는 방식이 안전합니다.

## 모델: BEATs
Microsoft UNILM의 BEATs encoder checkpoint를 불러온 뒤 mean-pooled representation 위에 위험 소리 classifier head를 학습합니다. BEATs checkpoint는 대용량이며 OneDrive/외부 파일이므로, 신뢰 가능한 출처에서 받은 checkpoint 경로를 지정하세요.

In [11]:
# Colab 공통 설치 셀
!pip -q install librosa soundfile pandas scikit-learn tqdm kaggle matplotlib

In [12]:
# BEATs 공식 구현 클론
import os, sys, subprocess
from pathlib import Path
repo = Path('/content/unilm')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/microsoft/unilm.git', str(repo)], check=True)
sys.path.append('/content/unilm/beats')

In [13]:
import os
import re
import json
import math
import random
import zipfile
import shutil
import subprocess
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =========================================================
# 1) 데이터셋 설정
# =========================================================
DATA_ROOT = Path('/content/sound_data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 회의록 사진/보고서에 있는 데이터셋을 모두 지원합니다.
# ESC-50은 자동 다운로드, UrbanSound8K는 Kaggle 계정이 있으면 자동 다운로드,
# FSD50K/AudioSet/AI Hub는 크기·라이선스·로그인 이슈 때문에 Google Drive에 받아둔 경로를 연결하는 방식입니다.
DATASETS_TO_USE = [
    'esc50',
    'urbansound8k',
    'fsd50k',
    'audioset',
    'aihub_noise',
]

USE_KAGGLE_URBANSOUND8K = False  # Kaggle API 토큰(kaggle.json)을 넣었다면 True로 바꾸세요.

# 큰 데이터셋은 Drive에 직접 다운로드한 뒤 아래 경로를 맞추세요.
FSD50K_ROOT = Path('/content/drive/MyDrive/datasets/FSD50K')
AUDIOSET_ROOT = Path('/content/drive/MyDrive/datasets/AudioSet_prepared')
AIHUB_NOISE_ROOT = Path('/content/drive/MyDrive/datasets/AIHub_noise_speech')

# Colab 빠른 실험용 제한. 최종 실험 때 늘리거나 None으로 바꾸세요.
MAX_SAMPLES_PER_CLASS = 600
MAX_OTHER_SAMPLES = 600
MIN_SAMPLES_PER_CLASS = 5

# 위험/주의 분류 타깃. 실제 서비스에서는 필요에 따라 baby_cry/doorbell을 별도 "주의" 레벨로 매핑하면 됩니다.
TARGET_LABELS = [
    'car_horn',
    'siren',
    'glass_breaking',
    'explosion_or_gunshot',
    'construction_or_machine',
    'fire_alarm',
    'baby_cry',
    'doorbell_or_bell',
    'other',
]
DANGER_LABELS = {
    'car_horn', 'siren', 'glass_breaking', 'explosion_or_gunshot',
    'construction_or_machine', 'fire_alarm'
}

# 라벨 매핑 우선순위: 여러 라벨이 동시에 있을 때 위험도가 높은 쪽을 선택합니다.
LABEL_PRIORITY = [
    'explosion_or_gunshot',
    'glass_breaking',
    'fire_alarm',
    'siren',
    'car_horn',
    'construction_or_machine',
    'baby_cry',
    'doorbell_or_bell',
    'other',
]

KEYWORD_RULES = [
    ('explosion_or_gunshot', [
        'explosion', 'explode', 'blast', 'gun_shot', 'gunshot', 'gun shot', 'gunfire', 'fireworks', 'firework',
        '폭발', '폭발음', '총성', '총소리', '사격', '폭죽'
    ]),
    ('glass_breaking', [
        'glass_breaking', 'glass breaking', 'glass_break', 'breaking_glass', 'shatter', 'glass',
        '유리', '유리깨짐', '유리 깨', '파손'
    ]),
    ('fire_alarm', [
        'fire_alarm', 'smoke_alarm', 'smoke detector', 'alarm', 'beep', 'buzzer',
        '화재경보', '화재 경보', '경보기', '경보음', '경보'
    ]),
    ('siren', [
        'siren', 'civil_defense_siren', 'emergency_vehicle', 'ambulance', 'police_siren',
        '싸이렌', '사이렌', '구급차', '경찰차'
    ]),
    ('car_horn', [
        'car_horn', 'vehicle_horn', 'air_horn', 'horn', 'honk',
        '경적', '경적소음', '클락션', '자동차 경적'
    ]),
    ('construction_or_machine', [
        'jackhammer', 'drilling', 'drill', 'chainsaw', 'hand_saw', 'saw', 'hammer', 'construction', 'machine',
        '착암', '드릴', '전동해머', '해머드릴', '기계톱', '절단기', '공사', '공장', '프레스'
    ]),
    ('baby_cry', [
        'crying_baby', 'baby_cry', 'baby crying', 'infant cry',
        '아기 울음', '아이 울음', '영아', '울음소리'
    ]),
    ('doorbell_or_bell', [
        'doorbell', 'door_bell', 'bell', 'door_knock', 'knock', 'church_bells',
        '초인종', '벨소리', '노크', '문 두드림', '방문'
    ]),
]

LABEL2ID = {label: i for i, label in enumerate(TARGET_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

def normalize_text(x):
    x = str(x).strip().lower()
    x = x.replace('-', '_').replace('/', '_').replace('\\', '_')
    x = re.sub(r'\s+', ' ', x)
    return x

def canonicalize_label(raw_label):
    """원본 데이터셋 라벨을 프로젝트 타깃 라벨로 변환합니다."""
    s = normalize_text(raw_label)
    for canonical, keys in KEYWORD_RULES:
        for key in keys:
            if normalize_text(key) in s:
                return canonical
    return 'other'

def pick_best_label(raw_labels):
    mapped = [canonicalize_label(x) for x in raw_labels if str(x).strip()]
    if not mapped:
        return 'other'
    for label in LABEL_PRIORITY:
        if label in mapped:
            return label
    return 'other'

def maybe_download_esc50(root=DATA_ROOT):
    target = root / 'ESC-50-master'
    if target.exists():
        return target
    url = 'https://github.com/karolpiczak/ESC-50/archive/master.zip'
    zip_path = root / 'ESC-50-master.zip'
    print(f'[ESC-50] downloading: {url}')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(root)
    zip_path.unlink(missing_ok=True)
    return target

def maybe_download_urbansound8k(root=DATA_ROOT):
    target = root / 'UrbanSound8K'
    if target.exists():
        return target
    if not USE_KAGGLE_URBANSOUND8K:
        print('[UrbanSound8K] USE_KAGGLE_URBANSOUND8K=False라서 자동 다운로드를 건너뜁니다.')
        return target
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    if not kaggle_json.exists():
        print('[UrbanSound8K] Kaggle 토큰이 없습니다. Colab에 kaggle.json을 업로드한 뒤 ~/.kaggle/kaggle.json에 배치하세요.')
        return target
    target.mkdir(parents=True, exist_ok=True)
    print('[UrbanSound8K] downloading from Kaggle: chrisfilo/urbansound8k')
    subprocess.run(['kaggle', 'datasets', 'download', '-d', 'chrisfilo/urbansound8k', '-p', str(target)], check=True)
    for zp in target.glob('*.zip'):
        with zipfile.ZipFile(zp, 'r') as zf:
            zf.extractall(target)
        zp.unlink(missing_ok=True)
    return target

def load_esc50(root):
    csv_candidates = list(Path(root).rglob('esc50.csv'))
    if not csv_candidates:
        return []
    meta_path = csv_candidates[0]
    base = meta_path.parent.parent
    audio_dir = base / 'audio'
    meta = pd.read_csv(meta_path)
    samples = []
    for _, row in meta.iterrows():
        raw = str(row.get('category', ''))
        label = canonicalize_label(raw)
        path = audio_dir / str(row['filename'])
        if path.exists():
            samples.append({'path': str(path), 'label': label, 'source': 'ESC-50', 'raw_label': raw})
    return samples

def load_urbansound8k(root):
    root = Path(root)
    csv_candidates = list(root.rglob('UrbanSound8K.csv'))
    if not csv_candidates:
        return []
    meta_path = csv_candidates[0]
    # 보통 .../UrbanSound8K/metadata/UrbanSound8K.csv 구조입니다.
    base = meta_path.parent.parent if meta_path.parent.name.lower() == 'metadata' else meta_path.parent
    meta = pd.read_csv(meta_path)
    samples = []
    for _, row in meta.iterrows():
        raw = str(row.get('class', row.get('classID', '')))
        label = canonicalize_label(raw)
        fname = str(row['slice_file_name'])
        fold = f"fold{int(row['fold'])}" if 'fold' in row else ''
        candidates = [base / 'audio' / fold / fname, base / fold / fname]
        path = next((p for p in candidates if p.exists()), None)
        if path is None:
            matches = list(base.rglob(fname))
            path = matches[0] if matches else None
        if path and path.exists():
            samples.append({'path': str(path), 'label': label, 'source': 'UrbanSound8K', 'raw_label': raw})
    return samples

def load_fsd50k(root):
    root = Path(root)
    if not root.exists():
        return []
    csvs = []
    for name in ['dev.csv', 'eval.csv']:
        csvs.extend(root.rglob(name))
    samples = []
    for csv_path in csvs:
        try:
            meta = pd.read_csv(csv_path)
        except Exception as e:
            print(f'[FSD50K] CSV read fail: {csv_path} / {e}')
            continue
        split = 'dev' if 'dev' in csv_path.name.lower() else 'eval'
        audio_dirs = list(root.rglob(f'FSD50K.{split}_audio')) + list(root.rglob(f'{split}_audio'))
        audio_dir = audio_dirs[0] if audio_dirs else root
        fname_col = 'fname' if 'fname' in meta.columns else meta.columns[0]
        labels_col = 'labels' if 'labels' in meta.columns else None
        if labels_col is None:
            continue
        for _, row in meta.iterrows():
            fname = str(row[fname_col])
            raw_labels = str(row[labels_col]).split(',')
            label = pick_best_label(raw_labels)
            candidates = [audio_dir / f'{fname}.wav', audio_dir / fname]
            path = next((p for p in candidates if p.exists()), None)
            if path is None:
                matches = list(root.rglob(f'{fname}.wav'))
                path = matches[0] if matches else None
            if path and path.exists():
                samples.append({'path': str(path), 'label': label, 'source': 'FSD50K', 'raw_label': ','.join(raw_labels)})
    return samples

def load_audiofolder_by_parent(root, source_name):
    """root 아래 wav/flac/mp3 파일을 찾고, 상위 폴더명을 라벨로 사용합니다. AudioSet을 직접 wav로 정리한 경우 유용합니다."""
    root = Path(root)
    if not root.exists():
        return []
    exts = ['*.wav', '*.flac', '*.mp3', '*.ogg', '*.m4a']
    files = []
    for ext in exts:
        files.extend(root.rglob(ext))
    samples = []
    for path in files:
        raw = path.parent.name
        label = canonicalize_label(raw)
        samples.append({'path': str(path), 'label': label, 'source': source_name, 'raw_label': raw})
    return samples

def flatten_values(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield str(k)
            yield from flatten_values(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from flatten_values(v)
    else:
        yield str(obj)

def load_aihub_noise(root):
    """AI Hub JSON 구조가 세부 버전마다 달라서, JSON 전체 텍스트에서 소음 라벨 키워드를 탐색합니다."""
    root = Path(root)
    if not root.exists():
        return []
    samples = []
    json_files = list(root.rglob('*.json'))
    used_audio = set()
    for jp in tqdm(json_files, desc='AI Hub JSON scan'):
        try:
            data = json.loads(jp.read_text(encoding='utf-8'))
        except Exception:
            try:
                data = json.loads(jp.read_text(encoding='cp949'))
            except Exception:
                continue
        raw_text = ' '.join(flatten_values(data))
        label = canonicalize_label(raw_text)
        # 같은 stem 또는 같은 폴더의 오디오 파일을 찾습니다.
        candidates = []
        for ext in ['.wav', '.flac', '.mp3', '.m4a']:
            candidates.append(jp.with_suffix(ext))
        if not any(p.exists() for p in candidates):
            candidates.extend(list(jp.parent.glob('*.wav')))
            candidates.extend(list(jp.parent.glob('*.flac')))
        path = next((p for p in candidates if p.exists() and str(p) not in used_audio), None)
        if path is not None:
            used_audio.add(str(path))
            samples.append({'path': str(path), 'label': label, 'source': 'AIHub_noise', 'raw_label': raw_text[:160]})
    # JSON이 없다면 폴더명 기반으로라도 로딩합니다.
    if not samples:
        samples = load_audiofolder_by_parent(root, 'AIHub_noise')
    return samples

def limit_samples(df):
    pieces = []
    for label, g in df.groupby('label'):
        cap = MAX_OTHER_SAMPLES if label == 'other' else MAX_SAMPLES_PER_CLASS
        if cap is not None and len(g) > cap:
            g = g.sample(cap, random_state=SEED)
        pieces.append(g)
    return pd.concat(pieces, ignore_index=True) if pieces else df

def safe_train_test_split(df, test_size, stratify_col='label'):
    counts = df[stratify_col].value_counts()
    stratify = df[stratify_col] if len(counts) > 1 and counts.min() >= 2 else None
    return train_test_split(df, test_size=test_size, random_state=SEED, stratify=stratify)

def build_metadata():
    all_samples = []

    if 'esc50' in DATASETS_TO_USE:
        esc_root = maybe_download_esc50(DATA_ROOT)
        s = load_esc50(esc_root)
        print(f'[ESC-50] {len(s)} samples')
        all_samples.extend(s)

    if 'urbansound8k' in DATASETS_TO_USE:
        urban_root = maybe_download_urbansound8k(DATA_ROOT)
        s = load_urbansound8k(urban_root)
        print(f'[UrbanSound8K] {len(s)} samples')
        all_samples.extend(s)

    if 'fsd50k' in DATASETS_TO_USE:
        s = load_fsd50k(FSD50K_ROOT)
        print(f'[FSD50K] {len(s)} samples from {FSD50K_ROOT}')
        all_samples.extend(s)

    if 'audioset' in DATASETS_TO_USE:
        # AudioSet은 공식적으로 YouTube segment/ontology 중심이라, wav로 준비한 폴더를 읽는 형태로 제공합니다.
        s = load_audiofolder_by_parent(AUDIOSET_ROOT, 'AudioSet_prepared')
        print(f'[AudioSet prepared audiofolder] {len(s)} samples from {AUDIOSET_ROOT}')
        all_samples.extend(s)

    if 'aihub_noise' in DATASETS_TO_USE:
        s = load_aihub_noise(AIHUB_NOISE_ROOT)
        print(f'[AI Hub noise] {len(s)} samples from {AIHUB_NOISE_ROOT}')
        all_samples.extend(s)

    df = pd.DataFrame(all_samples)
    if df.empty:
        raise RuntimeError('로드된 오디오 샘플이 없습니다. ESC-50 다운로드 상태 또는 Drive/Kaggle 경로를 확인하세요.')

    df = df.drop_duplicates('path').reset_index(drop=True)
    df = df[df['label'].isin(TARGET_LABELS)].reset_index(drop=True)
    df = limit_samples(df)

    counts = df['label'].value_counts()
    keep = counts[counts >= MIN_SAMPLES_PER_CLASS].index.tolist()
    dropped = sorted(set(df['label']) - set(keep))
    if dropped:
        print('[WARN] 샘플 수가 너무 적어 제외되는 클래스:', dropped)
        df = df[df['label'].isin(keep)].reset_index(drop=True)

    active_labels = [x for x in TARGET_LABELS if x in set(df['label'])]
    label2id = {label: i for i, label in enumerate(active_labels)}
    id2label = {i: label for label, i in label2id.items()}
    df['label_id'] = df['label'].map(label2id).astype(int)

    train_val_df, test_df = safe_train_test_split(df, test_size=0.15)
    train_df, val_df = safe_train_test_split(train_val_df, test_size=0.1765)  # 약 70/15/15

    print('\n[전체 라벨 분포]')
    print(df.groupby(['label', 'source']).size().unstack(fill_value=0))
    print('\n[split sizes]', {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)})
    print('[active labels]', active_labels)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True), active_labels, label2id, id2label

def load_audio_np(path, sr=16000, clip_seconds=5.0, random_crop=False):
    """오디오를 mono float32 [-1, 1] 근사 범위로 읽고 고정 길이로 crop/pad합니다."""
    y, _ = librosa.load(path, sr=sr, mono=True)
    if not np.isfinite(y).all():
        y = np.nan_to_num(y)
    target_len = int(sr * clip_seconds)
    if len(y) == 0:
        y = np.zeros(target_len, dtype=np.float32)
    if len(y) > target_len:
        if random_crop:
            start = random.randint(0, len(y) - target_len)
        else:
            start = max(0, (len(y) - target_len) // 2)
        y = y[start:start + target_len]
    elif len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    y = y.astype(np.float32)
    peak = np.max(np.abs(y)) if len(y) else 0
    if peak > 1.0:
        y = y / peak
    return y

def augment_waveform_np(y, sr):
    y = y.copy()
    if random.random() < 0.50:
        gain_db = random.uniform(-6.0, 6.0)
        y *= 10 ** (gain_db / 20.0)
    if random.random() < 0.35:
        shift = random.randint(-sr // 5, sr // 5)
        y = np.roll(y, shift)
    if random.random() < 0.35:
        noise_scale = random.uniform(0.0005, 0.006)
        y = y + np.random.randn(len(y)).astype(np.float32) * noise_scale
    return np.clip(y, -1.0, 1.0).astype(np.float32)

def make_class_weights(train_df, active_labels):
    labels = np.array([active_labels[i] for i in train_df['label_id'].values])
    classes = np.array(active_labels)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    return {i: float(w) for i, w in enumerate(weights)}

def save_labels_json(path, id2label):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({int(k): v for k, v in id2label.items()}, ensure_ascii=False, indent=2), encoding='utf-8')
    print('saved:', path)

In [14]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

class AudioClipDataset(Dataset):
    def __init__(self, df, label2id, sample_rate, clip_seconds, augment=False):
        self.df = df.reset_index(drop=True)
        self.label2id = label2id
        self.sample_rate = sample_rate
        self.clip_seconds = clip_seconds
        self.augment = augment
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = load_audio_np(row['path'], sr=self.sample_rate, clip_seconds=self.clip_seconds, random_crop=self.augment)
        if self.augment:
            y = augment_waveform_np(y, self.sample_rate)
        label = int(row['label_id'])
        return torch.from_numpy(y), torch.tensor(label, dtype=torch.long)

def make_weighted_sampler(train_df, num_labels):
    counts = train_df['label_id'].value_counts().to_dict()
    weights = [1.0 / counts[int(y)] for y in train_df['label_id'].values]
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

def make_loaders(train_df, val_df, test_df, label2id, sample_rate, clip_seconds, batch_size=16):
    train_ds = AudioClipDataset(train_df, label2id, sample_rate, clip_seconds, augment=True)
    val_ds = AudioClipDataset(val_df, label2id, sample_rate, clip_seconds, augment=False)
    test_ds = AudioClipDataset(test_df, label2id, sample_rate, clip_seconds, augment=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=make_weighted_sampler(train_df, len(label2id)), num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader

def run_torch_training(model, train_loader, val_loader, active_labels, class_weights, epochs=5, lr=1e-4, save_path='/content/best_model.pt'):
    model = model.to(DEVICE)
    weight = torch.tensor([class_weights.get(i, 1.0) for i in range(len(active_labels))], dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weight)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    best_f1 = -1
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for wav, labels in tqdm(train_loader, desc=f'train epoch {epoch}'):
            wav = wav.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                logits = model(wav)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * wav.size(0)

        val_metrics = evaluate_torch_model(model, val_loader, active_labels, verbose=False)
        train_loss = total_loss / max(1, len(train_loader.dataset))
        print(f"epoch={epoch} train_loss={train_loss:.4f} val_acc={val_metrics['accuracy']:.4f} val_macro_f1={val_metrics['macro_f1']:.4f}")
        if val_metrics['macro_f1'] > best_f1:
            best_f1 = val_metrics['macro_f1']
            torch.save({'model': model.state_dict(), 'active_labels': active_labels}, save_path)
            print('  saved best:', save_path)
    return model

@torch.no_grad()
def evaluate_torch_model(model, loader, active_labels, verbose=True):
    model.eval()
    y_true, y_pred = [], []
    for wav, labels in tqdm(loader, desc='eval', leave=False):
        wav = wav.to(DEVICE)
        logits = model(wav)
        pred = logits.argmax(dim=-1).cpu().numpy().tolist()
        y_pred.extend(pred)
        y_true.extend(labels.numpy().tolist())
    acc = accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    if verbose:
        print(classification_report(y_true, y_pred, target_names=active_labels, zero_division=0))
        print('confusion_matrix:\n', confusion_matrix(y_true, y_pred))
    return {'accuracy': acc, 'macro_f1': macro, 'y_true': y_true, 'y_pred': y_pred}

@torch.no_grad()
def predict_file_torch(model, audio_path, active_labels, sample_rate, clip_seconds):
    model.eval()
    y = load_audio_np(audio_path, sr=sample_rate, clip_seconds=clip_seconds, random_crop=False)
    wav = torch.from_numpy(y).unsqueeze(0).to(DEVICE)
    logits = model(wav)
    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
    order = probs.argsort()[::-1]
    return [(active_labels[i], float(probs[i])) for i in order[:5]]

DEVICE = cuda


## 데이터셋 로딩

In [15]:
train_df, val_df, test_df, active_labels, label2id, id2label = build_metadata()
class_weights = make_class_weights(train_df, active_labels)
print('class_weights:', class_weights)

[ESC-50] 2000 samples
[UrbanSound8K] USE_KAGGLE_URBANSOUND8K=False라서 자동 다운로드를 건너뜁니다.
[UrbanSound8K] 0 samples
[FSD50K] 0 samples from /content/drive/MyDrive/datasets/FSD50K
[AudioSet prepared audiofolder] 0 samples from /content/drive/MyDrive/datasets/AudioSet_prepared
[AI Hub noise] 0 samples from /content/drive/MyDrive/datasets/AIHub_noise_speech

[전체 라벨 분포]
source                   ESC-50
label                          
baby_cry                     40
car_horn                     40
construction_or_machine     120
doorbell_or_bell             80
explosion_or_gunshot         40
fire_alarm                   40
glass_breaking               40
other                       600
siren                        40

[split sizes] {'train': 727, 'val': 157, 'test': 156}
[active labels] ['car_horn', 'siren', 'glass_breaking', 'explosion_or_gunshot', 'construction_or_machine', 'fire_alarm', 'baby_cry', 'doorbell_or_bell', 'other']
class_weights: {0: 2.884920634920635, 1: 2.884920634920635, 2: 2.884

## BEATs checkpoint 준비
공식 README의 OneDrive 링크에서 `BEATs_iter3_plus_AS2M.pt` 또는 AudioSet fine-tuned checkpoint를 받아 `/content/BEATs_iter3_plus_AS2M.pt`에 두는 방식을 권장합니다. 아래 셀은 Colab 실행 편의를 위해 Hugging Face Hub에 공개된 checkpoint를 fallback으로 받을 수 있게 해둔 선택 코드입니다.

In [16]:
# 1) 권장: 공식 README의 BEATs_iter3+ (AS2M) pre-trained checkpoint를 자동 다운로드 시도
BEATS_CKPT_PATH = Path('/content/BEATs_iter3_plus_AS2M.pt')

# Remove corrupt checkpoint if it is too small (e.g., less than 1MB)
if BEATS_CKPT_PATH.exists() and BEATS_CKPT_PATH.stat().st_size < 1000000:
    print(f"Removing corrupt checkpoint: {BEATS_CKPT_PATH}")
    BEATS_CKPT_PATH.unlink(missing_ok=True)

OFFICIAL_BEATS_ONEDRIVE_URL = 'https://1drv.ms/u/s!AqeByhGUtINrgcpke6_lRSZEKD5j2Q?download=1'
USE_OFFICIAL_ONEDRIVE_DOWNLOAD = False

if (not BEATS_CKPT_PATH.exists()) and USE_OFFICIAL_ONEDRIVE_DOWNLOAD:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onedrivedownloader'], check=True)
    from onedrivedownloader import download
    try:
        download(OFFICIAL_BEATS_ONEDRIVE_URL, filename=str(BEATS_CKPT_PATH), unzip=False, force_download=False)
    except Exception as e:
        print('[WARN] 공식 OneDrive 자동 다운로드 실패:', repr(e))

# 2) 선택: Colab 빠른 실행용 fallback. 외부 torch 외부 pickle 파일이므로 신뢰 가능한 환경에서만 사용하세요.
USE_HF_FALLBACK_CHECKPOINT = True
if (not BEATS_CKPT_PATH.exists()) and USE_HF_FALLBACK_CHECKPOINT:
    print("Downloading BEATs checkpoint from Hugging Face Hub (nsivaku/nithin_checkpoints).")
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import hf_hub_download
    import shutil
    ckpt = hf_hub_download(repo_id='nsivaku/nithin_checkpoints', filename='BEATs_iter3_plus_AS2M.pt')
    shutil.copy(ckpt, BEATS_CKPT_PATH)

if not BEATS_CKPT_PATH.exists():
    raise FileNotFoundError(
        'BEATs checkpoint가 없습니다. Microsoft UNILM BEATs README의 checkpoint를 직접 다운로드해서 '
        '/content/BEATs_iter3_plus_AS2M.pt 경로에 두거나 USE_HF_FALLBACK_CHECKPOINT=True로 바꾸세요.'
    )
print('BEATs checkpoint:', BEATS_CKPT_PATH)

BEATs checkpoint: /content/BEATs_iter3_plus_AS2M.pt


## BEATs 모델 구성

In [17]:
import torch
from torch import nn
from BEATs import BEATs, BEATsConfig

SAMPLE_RATE = 16000
CLIP_SECONDS = 10.0
BATCH_SIZE = 4
EPOCHS = 5
LR = 1e-4
FREEZE_BACKBONE = True

class BEATsDangerClassifier(nn.Module):
    def __init__(self, checkpoint_path, num_classes, freeze_backbone=True):
        super().__init__()
        try:
            checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        except TypeError:
            checkpoint = torch.load(checkpoint_path, map_location='cpu')
        cfg = BEATsConfig(checkpoint['cfg'])
        # fine-tuned checkpoint를 쓰더라도 predictor는 제거하고 representation extractor로 씁니다.
        cfg.finetuned_model = False
        self.beats = BEATs(cfg)
        self.beats.load_state_dict(checkpoint['model'], strict=False)
        if freeze_backbone:
            for p in self.beats.parameters():
                p.requires_grad = False
        self.head = nn.Sequential(
            nn.LayerNorm(cfg.encoder_embed_dim),
            nn.Dropout(0.30),
            nn.Linear(cfg.encoder_embed_dim, num_classes),
        )
    def forward(self, waveform):
        padding_mask = torch.zeros(waveform.shape, dtype=torch.bool, device=waveform.device)
        feats, _ = self.beats.extract_features(waveform, padding_mask=padding_mask)
        pooled = feats.mean(dim=1)
        return self.head(pooled)

model = BEATsDangerClassifier(BEATS_CKPT_PATH, num_classes=len(active_labels), freeze_backbone=FREEZE_BACKBONE)
print(model)

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


BEATsDangerClassifier(
  (beats): BEATs(
    (post_extract_proj): Linear(in_features=512, out_features=768, bias=True)
    (patch_embedding): Conv2d(1, 512, kernel_size=(16, 16), stride=(16, 16), bias=False)
    (dropout_input): Dropout(p=0.1, inplace=False)
    (encoder): TransformerEncoder(
      (pos_conv): Sequential(
        (0): Conv1d(768, 768, kernel_size=(128,), stride=(1,), padding=(64,), groups=16)
        (1): SamePad()
        (2): GELU(approximate='none')
      )
      (layers): ModuleList(
        (0): TransformerSentenceEncoderLayer(
          (self_attn): MultiheadAttention(
            (dropout_module): Dropout(p=0.1, inplace=False)
            (relative_attention_bias): Embedding(320, 12)
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_featur

## 학습 / 평가

In [18]:
train_loader, val_loader, test_loader = make_loaders(train_df, val_df, test_df, label2id, SAMPLE_RATE, CLIP_SECONDS, batch_size=BATCH_SIZE)
model = run_torch_training(
    model, train_loader, val_loader, active_labels, class_weights,
    epochs=EPOCHS, lr=LR, save_path='/content/beats_danger_best.pt'
)

ckpt = torch.load('/content/beats_danger_best.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model'])
metrics = evaluate_torch_model(model, test_loader, active_labels, verbose=True)
print(metrics)

/tmp/ipykernel_29206/3121388422.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))


train epoch 1:   0%|          | 0/182 [00:00<?, ?it/s]

/tmp/ipykernel_29206/3121388422.py:56: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):


eval:   0%|          | 0/40 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680><function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()    
self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():<function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680><function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680>

Traceback (most recent call last):
Traceback (most recent call last):
  File "

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680>
Exception ignored in: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680>    
self._shutdown_workers()
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
        <function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680>
Exception ignored in: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680>    
self._shutdown_workers()
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/

epoch=5 train_loss=nan val_acc=0.0573 val_macro_f1=0.0356


eval:   0%|          | 0/39 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680>Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680>
<function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680>Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
<function _MultiProcessingDataLoaderIter.__del__ at 0x7ed6d9f24680>Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
Traceback (most re

                         precision    recall  f1-score   support

               car_horn       0.03      0.17      0.05         6
                  siren       0.08      0.67      0.14         6
         glass_breaking       0.00      0.00      0.00         6
   explosion_or_gunshot       0.00      0.00      0.00         6
construction_or_machine       0.08      0.06      0.06        18
             fire_alarm       0.00      0.00      0.00         6
               baby_cry       0.00      0.00      0.00         6
       doorbell_or_bell       0.00      0.00      0.00        12
                  other       1.00      0.01      0.02        90

               accuracy                           0.04       156
              macro avg       0.13      0.10      0.03       156
           weighted avg       0.59      0.04      0.03       156

confusion_matrix:
 [[ 1  5  0  0  0  0  0  0  0]
 [ 0  4  0  2  0  0  0  0  0]
 [ 0  2  0  0  4  0  0  0  0]
 [ 0  0  0  0  0  0  1  5  0]
 [ 8  6  0  1

## 파일 하나 예측 / 저장

In [19]:
sample_path = test_df.iloc[0]['path']
print(sample_path, test_df.iloc[0]['label'])
print(predict_file_torch(model, sample_path, active_labels, SAMPLE_RATE, CLIP_SECONDS))

save_labels_json('/content/beats_labels.json', id2label)
torch.save({'model': model.state_dict(), 'active_labels': active_labels}, '/content/beats_danger_final.pt')
print('saved /content/beats_danger_final.pt')

/content/sound_data/ESC-50-master/audio/4-204115-A-39.wav glass_breaking
[('construction_or_machine', 0.14857743680477142), ('car_horn', 0.142691969871521), ('baby_cry', 0.13628371059894562), ('siren', 0.1355745792388916), ('explosion_or_gunshot', 0.12705187499523163)]
saved: /content/beats_labels.json
[('construction_or_machine', 0.14857743680477142), ('car_horn', 0.142691969871521), ('baby_cry', 0.13628371059894562), ('siren', 0.1355745792388916), ('explosion_or_gunshot', 0.12705187499523163)]
saved: /content/beats_labels.json
saved /content/beats_danger_final.pt
saved /content/beats_danger_final.pt
